# ACTO SDK: Basic Proof Creation

This notebook demonstrates how to create and verify execution proofs using the ACTO SDK.

## Installation

```bash
pip install actobotics
```

> **Note:** The package is called `actobotics` on PyPI, but you import it as `acto` in Python.


In [ ]:
from datetime import datetime

from acto.crypto import KeyPair
from acto.proof import create_proof
from acto.client import ACTOClient
from acto.telemetry.models import TelemetryBundle, TelemetryEvent


## Generate a Keypair


In [ ]:
# Generate a new Ed25519 keypair
keypair = KeyPair.generate()
print(f"Public key: {keypair.public_key_b64}")


## Create Telemetry Bundle


In [ ]:
# Create sample telemetry data
telemetry_events = [
    TelemetryEvent(
        ts=datetime.now().isoformat(),
        topic="lidar",
        data={"distance": 1.5, "angle": 45.0}
    ),
    TelemetryEvent(
        ts=datetime.now().isoformat(),
        topic="camera",
        data={"image_id": "img_001", "detections": 3}
    )
]

# Create telemetry bundle
bundle = TelemetryBundle(
    task_id="cleaning-run-001",
    robot_id="robot-001",
    run_id="run-001",
    events=telemetry_events
)

print(f"Created bundle for task: {bundle.task_id}")


## Create Proof


In [ ]:
# Create signed proof envelope
envelope = create_proof(
    bundle,
    keypair.private_key_b64,
    keypair.public_key_b64
)

print("Proof created!")
print(f"Payload hash: {envelope.payload.payload_hash}")
print(f"Task ID: {envelope.payload.subject.task_id}")


## Verify Proof via API

Proof verification must be done through the ACTO API. Get your API key at [https://acto-production.up.railway.app/dashboard](https://https://acto-production.up.railway.app/dashboard).


In [ ]:
# Initialize the API client
# Replace with your actual API key and wallet address from the dashboard
API_KEY = "YOUR_API_KEY"  # Get from https://https://acto-production.up.railway.app/dashboard
WALLET_ADDRESS = "YOUR_WALLET_ADDRESS"

client = ACTOClient(
    api_key=API_KEY,
    wallet_address=WALLET_ADDRESS
)

# Verify the proof via API
try:
    result = client.verify(envelope)
    print(f"✓ Proof is valid: {result.valid}")
    print(f"  Reason: {result.reason}")
except Exception as e:
    print(f"✗ Proof verification failed: {e}")
    print("  Make sure to set your API_KEY and WALLET_ADDRESS above!")


## Save Proof to File


In [ ]:
from pathlib import Path

# Save proof to file
output_path = Path("proof.json")
output_path.write_text(envelope.model_dump_json(indent=2))
print(f"Proof saved to: {output_path}")
